# variableMeanDriftingGrating

Protocol-specific analysis. `demos/meaAnalysisMain.ipynb` is the shared front half — it finds datasets, builds a pipeline and checks that the noise chunk and the protocol datafile describe the same cells. That notebook deliberately stops before interpreting conditions, because that is where protocols stop resembling each other. This is where it picks up.

The protocol drifts a sinewave grating at a fixed temporal frequency while alternating two things across epochs: the **background mean intensity** and the **bar width**. So the design is mean × bar width, and every epoch belongs to one cell of that grid.

**This notebook loads its own data.** Copy the experiment and datafile out of the main notebook's §6 printout into the constants below and run from the top — nothing is inherited from another kernel. If you *are* in the same kernel and already have a `pipeline`, the load cell will reuse it rather than rebuild.

The order here is deliberate: read the protocol source, get the condition axes from it, then clean — **epochs first, then cells**. Cleaning the other way round scores every cell against a stretch of block you were going to discard anyway, which reports a property of the block as a property of each cell.

## 1. Setup and the dataset

`PROTOCOL_NAME` is the full dotted name as the database stores it — §2 uses it to find the MATLAB source, so it has to be the real one, not the search fragment.

`create_mea_pipeline` is the same call the main notebook's §6 makes. Pinning `ANALYSIS_CHUNK` is optional; leave it `None` and the nearest noise chunk with a typing file is chosen, by the same rule the main notebook uses.

In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# <-- EDIT ME: from the main notebook's §6 printout.
EXP_NAME       = '20230502C'
DATAFILE_NAME  = 'data017'
ANALYSIS_CHUNK = 'chunk2'     # None to let the pipeline pick

PROTOCOL_NAME = 'edu.washington.riekelab.chris.protocols.variableMeanDriftingGrating'
MAIN_TYPES    = ['OnP', 'OffP', 'OnM', 'OffM']

# Reuse a pipeline from the same kernel when it is already the right dataset;
# otherwise build one. Makes the notebook runnable standalone without
# rebuilding needlessly when it isn't.
_existing = globals().get('pipeline')
if (_existing is not None
        and getattr(_existing.resp, 'datafile_name', None) == DATAFILE_NAME
        and _existing.analysis_chunk.exp_name == EXP_NAME):
    print(f'Reusing the pipeline already in this kernel: {EXP_NAME}/{DATAFILE_NAME}')
else:
    pipeline = ra.create_mea_pipeline(EXP_NAME, DATAFILE_NAME,
                                      analysis_chunk_name = ANALYSIS_CHUNK)

stim_block     = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk

## 2. What the protocol declares

The MATLAB class is the authority on what the experiment did, and it is worth reading before trusting any assumption about the data. `ra.parse_protocol_source` finds the `.m` in the locally cloned package — the dotted protocol name maps straight onto MATLAB's package layout, so `edu.washington.riekelab.chris.protocols.X` is `+edu/+washington/+riekelab/+chris/+protocols/X.m` under `chris-package` — and reads two things out of it:

- the **`properties` block**, the protocol's parameters and their defaults, fixed for a whole epoch block;
- the **`epoch.addParameter(...)` calls** in `prepareEpoch`, which are exactly the parameters that vary from epoch to epoch. Those are the condition axes, and nothing else is.

It also prints the GitHub URL for the file, so the source of truth is one click away: <https://github.com/Rieke-Lab/chris-package/>.

Hidden properties are listed too but flagged. They are the rig plumbing and the working variables the protocol uses while preparing an epoch — you never set them, but they explain where a per-epoch value came from.

If the package isn't cloned under `PROTOCOL_REPOS_ROOT`, this prints where it looked and returns `None`; the rest of the notebook still runs off the recorded parameters.

In [ ]:
source = ra.parse_protocol_source(PROTOCOL_NAME)

print(f'{source.class_name}  <  {source.superclass}')
print(f'source : {source.path}')
print(f'github : {source.github_url}')
print(f'\nepoch-specific parameters (the condition axes): {source.epoch_parameters}\n')

display(source.parameters.query('not hidden')[['parameter', 'default', 'comment']])

### Declared vs recorded

`compare_with_block` puts the source beside what this block actually carries. Two things it catches, both of which bite silently otherwise:

**Defaults are not what ran.** The `.m` declares `meanIntensities = [0.05 0.5]` and `barWidths = [40 150]`, but the rig was set to different values on the day. The `recorded` column is what to believe.

**Names in the source are not always the names in the data.** This protocol calls `epoch.addParameter('currentBarWdith', ...)` — misspelled in the MATLAB — so `currentBarWdith` is the column name in the database. Analysis that reasonably reaches for `currentBarWidth` finds nothing and silently gets an all-`None` condition axis. `in_data` is the column that exposes this: a declared parameter marked `False` either wasn't recorded by this protocol version or is spelled differently in the data.

In [ ]:
params = ra.compare_with_block(source, stim_block)
display(params)

# The names to actually index with, taken from the source rather than guessed.
CONDITION_KEYS = [p for p in source.epoch_parameters
                  if params.set_index('parameter').loc[p, 'in_data']]
print('condition axes present in the data:', CONDITION_KEYS)

## 3. The condition grid

Read the per-epoch values with the names §2 just confirmed. `currentBackgroundScale`-style parameters are not always promoted to their own `df_epochs` column — whether they are depends on the protocol version that recorded the date — so this falls back to the raw `epoch_parameters` dicts rather than assuming a column exists.

The crosstab is the design as it actually ran: how many epochs landed in each cell of the mean × bar-width grid. An unbalanced grid here is worth noticing before it turns into an unbalanced comparison later — and it gets more unbalanced in §4, since trimming a range of epochs does not trim conditions evenly unless the range is a whole number of cycles.

In [ ]:
def epoch_param(df, key):
    """Per-epoch values for `key`, from its column if promoted, else the raw dicts."""
    if key in df.columns:
        return df[key].to_numpy()
    return np.array([d.get(key) for d in df['epoch_parameters']], dtype = object)


df_epochs  = stim_block.df_epochs
conditions = {key: epoch_param(df_epochs, key) for key in CONDITION_KEYS}

for key, values in conditions.items():
    print(f'{key:24s} levels: {sorted(set(values))}')

# The design as it ran, not as declared.
grid = pd.crosstab(pd.Series(conditions[CONDITION_KEYS[0]], name = CONDITION_KEYS[0]),
                   pd.Series(conditions[CONDITION_KEYS[1]], name = CONDITION_KEYS[1]))
print(f'\n{len(df_epochs)} epochs over the condition grid:')
display(grid)

## 4. Clean the epochs

The main notebook's spikes-per-epoch view showed the population rate is not constant across this block. This decides which stretch of it to analyze.

**Normalize within condition, or the experiment looks like dropout.** The mean intensity alternates every epoch, so raw population rate alternates with it — every other epoch sits at a fraction of the median purely by design. A threshold on raw rate would discard one entire condition and call it quality control. Passing `condition_values` divides each epoch by the median of the epochs sharing its condition, so a dim epoch is judged against other dim epochs.

**A contiguous range, not a mask.** A block goes bad by drifting, not by scattering, and a non-contiguous selection silently unbalances the conditions.

**Read the plot before accepting the answer.** On `20230502C/data017` the population rate rises roughly tenfold from the first epoch to the last, monotonically, in *both* conditions — that is adaptation, not cells dropping out. Trimming the early epochs is therefore a scientific choice about which part of the adaptation you want to analyze, not the removal of bad data. Lower `MIN_FRACTION` toward 0 to keep everything, or set `EPOCH_RANGE` by hand.

In [ ]:
MIN_FRACTION = 0.5   # <-- EDIT ME: fraction of a condition's median to count as usable

# Normalize against the alternating axis — the one that changes every epoch.
alternating = CONDITION_KEYS[0]

epoch_fit = ra.suggest_epoch_range(response_block, cell_types = MAIN_TYPES,
                                   minimum_n = 3, min_fraction = MIN_FRACTION,
                                   condition_values = conditions[alternating])

EPOCH_RANGE = (epoch_fit['start'], epoch_fit['stop'])   # <-- override by hand if you disagree
print(f'keeping epochs {EPOCH_RANGE[0]}–{EPOCH_RANGE[1] - 1} '
      f'({epoch_fit["n_kept"]} of {epoch_fit["n_epochs"]}), '
      f'normalized within {alternating}')

ra.plot_epoch_range(epoch_fit, condition_values = conditions[alternating],
                    title = f'{EXP_NAME}/{DATAFILE_NAME} — epochs kept')

# What the trim did to the balance of the design.
kept = slice(*EPOCH_RANGE)
grid_kept = pd.crosstab(pd.Series(conditions[CONDITION_KEYS[0]][kept], name = CONDITION_KEYS[0]),
                        pd.Series(conditions[CONDITION_KEYS[1]][kept], name = CONDITION_KEYS[1]))
print('\ncondition grid after trimming:')
display(grid_kept)

## 5. Clean the cells

Now that the epoch range is settled, judge cells on those epochs. `block_qc_metrics` takes `epoch_range` for exactly this reason: a cell scored over a stretch of dead block fails on `silent_run_max` and `drift_score` for a reason that has nothing to do with the cell.

The gates come from `ra.QCThresholds`, and two of the defaults matter here:

- **The firing-rate gate is adaptive.** The threshold is `min_rate_hz × epoch_duration_s`, and a cell passes when at least `min_frac_epochs_above_rate` of its epochs clear it. That scales across protocols with different stimulus durations without tuning — this one runs 60 s epochs.
- **`min_reliability_r` is off by default, and must stay off here.** Split-half PSTH correlation is only meaningful when every epoch presents the same stimulus. This protocol alternates conditions epoch by epoch, so split-half correlation compares bright trials against dim ones and reports a low number for a perfectly reliable cell. To use reliability, group epochs by condition first.

**Pass `t_end_ms`.** The rate gate divides spike counts by the epoch length, and with no length to divide by it comes back NaN — the comparison is then False for every cell and the entire block fails QC while looking like data so bad nothing survived. The epoch length here is the protocol's own `preTime + stimTime + tailTime`. `block_qc_metrics` will infer one from the latest spike in the block and say so, but that is a floor, short by however long the last epoch stayed quiet.\n\nAdjust the thresholds to taste and re-run; the summary reports how many cells each gate cost.

In [ ]:
# Epoch length from the protocol's own timing. Pass this explicitly: the rate
# gate divides by it, and without it every metric that depends on rate comes
# back NaN and the whole block fails QC silently.
block = stim_block.d_epoch_block_params
T_END_MS = sum(float(block.get(k, 0) or 0) for k in ('preTime', 'stimTime', 'tailTime'))
print(f'epoch length {T_END_MS / 1000:.0f} s  '
      f'(preTime {block.get("preTime")} + stimTime {block.get("stimTime")} '
      f'+ tailTime {block.get("tailTime")} ms)')

# "Alive in one condition" rather than "alive in all of them". With levels
# alternating epoch to epoch, a flat 0.8 demands the cell clear the rate gate
# in the DIM epochs too, which is a response requirement wearing a quality
# gate's clothes. 0.8 / n_levels asks for the same 80% within one condition's
# worth of epochs, and adapts if a protocol has three levels instead of two.
n_levels = len(set(conditions[alternating][slice(*EPOCH_RANGE)]))
MIN_FRAC_EPOCHS = round(0.8 / n_levels, 2)
print(f'{n_levels} levels of {alternating} in the kept epochs '
      f'-> rate gate needs {MIN_FRAC_EPOCHS:.0%} of epochs above threshold\n')

thresholds = ra.QCThresholds(
    min_rate_hz = 1.0,                             # per-epoch rate, scaled by epoch length
    min_frac_epochs_above_rate = MIN_FRAC_EPOCHS,  # ... in one condition's worth of epochs
    min_reliability_r = None,                      # off: conditions alternate, see above
)

qc = ra.block_qc_metrics(response_block, cell_types = MAIN_TYPES,
                         epoch_range = EPOCH_RANGE, t_end_ms = T_END_MS,
                         min_rate_hz = thresholds.min_rate_hz)
qc = ra.filter_cells_by_qc(qc, thresholds)

print(f'{int(qc["passes"].sum())} of {len(qc)} cells pass, '
      f'scored on epochs {EPOCH_RANGE[0]}\u2013{EPOCH_RANGE[1] - 1}\n')

summary = (qc.groupby('cell_type')
             .agg(n_cells = ('passes', 'size'), n_kept = ('passes', 'sum'),
                  median_rate_hz = ('mean_rate_hz', 'median'))
             .assign(frac_kept = lambda d: (d.n_kept / d.n_cells).round(2)))
display(summary.sort_values('n_cells', ascending = False))

GOOD_CELLS = qc.query('passes')['cell_id'].astype(int).tolist()

## 6. What the cleaning did

The same views as before, restricted to what survived, so the effect of both filters is visible rather than assumed.

The rasters are the direct check: draw the kept cells over the first and last of the **kept** epochs. If a row is still blank on one side, the gates were too loose. The dropped cells are worth a look too — a large group of them firing perfectly well means a threshold is wrong, not that the cells are.

The comparison table is the one to read before moving on. If a cell type lost most of its cells, any population claim about that type from here on rests on whatever is left, and the honest thing is to say so — or to loosen the gates and say that instead.

In [ ]:
# Kept vs dropped, side by side.
comparison = (qc.assign(group = np.where(qc['passes'], 'kept', 'dropped'))
                .groupby(['cell_type', 'group'])
                .agg(n = ('cell_id', 'size'),
                     median_rate_hz = ('mean_rate_hz', 'median'),
                     median_silent_frac = ('silent_trial_frac', 'median'))
                .round(2))
display(comparison)

# Kept cells only — cell_ids restricts the dropdown counts as well as the
# panels, so a label describes what is actually drawn. The epoch axis still
# spans the whole block on purpose: seeing the discarded stretch next to the
# kept one is how you check the trim was right.
print(f'\nRasters, kept cells only ({len(GOOD_CELLS)} of {len(qc)})')
ra.browse_epoch_rasters(response_block, cell_types = MAIN_TYPES, minimum_n = 1,
                        cell_ids = GOOD_CELLS, n_first = 3, n_last = 3);

## 7. Where to go from here

`EPOCH_RANGE`, `GOOD_CELLS` and `conditions` are the cleaned handles. Everything downstream should be written against those three rather than against the raw block, so the cleaning applies once and visibly.

The obvious next step for this protocol is the response as a function of the two condition axes — PSTHs per (mean intensity × bar width) cell of the grid, and a summary of how mean intensity shifts the grating response at each bar width. `ra.get_spike_xarr(response_block, cell_types=MAIN_TYPES)` gives a ragged (cell × epoch) array to build that on; slice it with `EPOCH_RANGE` and select cells with `GOOD_CELLS`.

Worth carrying forward, from what the cleaning showed: the tenfold rise in population rate across this block is monotonic and in both conditions. Any comparison between conditions is safe — they alternate, so both are sampled evenly across the trend — but a comparison between *early and late* epochs is confounded with it.